# Post-Redesign OOF and Ensemble Diagnostic

This notebook documents a post-redesign diagnostic of the base-model probability pipeline. It compares OOF, validation-ensemble, and label-blind locked-test-ensemble probability distributions without tuning, selecting, averaging, or ranking models.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, row-level predictions, checkpoints, fold models, or private run artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the post-redesign diagnostic

Imports descriptive-statistics and plotting utilities, defines the restricted project-root placeholder, specifies expected seeds and row counts, and declares the probability and meta-feature fields examined by the diagnostic.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import uuid

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
R1_06_ROOT = ROUND1_DIR / 'artifacts' / 'r1_06_repeated_seed_base_oof_v2'
OUTPUT_DIR = ROUND1_DIR / 'reports' / 'oof_ensemble_post_redesign_diagnostic'
FIGURE_DIR = ROUND1_DIR / 'figures' / 'oof_ensemble_post_redesign_diagnostic'

EXPECTED_SEEDS = [42, 52, 62]
LABEL_ORDER = ['negatif', 'netral', 'positif']
N_CLASSES = len(LABEL_ORDER)
EXPECTED_ROWS = {'oof_train': 4673, 'validation_ensemble': 1170, 'locked_test_ensemble': 1172}
EPSILON = 1e-12

SVM_PROB_COLUMNS = [f'svm_prob_{label}' for label in LABEL_ORDER]
BERT_PROB_COLUMNS = [f'bert_prob_{label}' for label in LABEL_ORDER]
META_FEATURES = SVM_PROB_COLUMNS + BERT_PROB_COLUMNS + [
    'svm_confidence', 'bert_confidence', 'svm_entropy', 'bert_entropy',
    'pred_disagreement', 'prob_disagreement',
]
SOURCE_FILES = {
    'oof_train': ('oof_train_base_probabilities.csv', 'oof_train_meta_features.csv'),
    'validation_ensemble': ('validation_ensemble_base_probabilities.csv', 'validation_meta_features.csv'),
    'locked_test_ensemble': ('locked_test_ensemble_base_probabilities_blind.csv', 'locked_test_meta_features_blind.csv'),
}

if not R1_06_ROOT.is_dir():
    raise FileNotFoundError(f'Base-model artifact tidak ditemukan: {R1_06_ROOT}')

print('Restricted project root placeholder:', PROJECT_ROOT)
print('Base-model source:', R1_06_ROOT)
print('Output       :', OUTPUT_DIR)
print('Seeds        :', EXPECTED_SEEDS)


### Define diagnostic validation utilities

Defines atomic writers, entropy and probability checks, source-key validation, base-to-meta feature consistency checks, and numeric distribution summaries.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8')
    os.replace(temporary, path)


def atomic_write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=True, default=float), encoding='utf-8')
    os.replace(temporary, path)


def normalized_entropy(probabilities: np.ndarray) -> np.ndarray:
    safe = np.clip(np.asarray(probabilities, dtype=float), EPSILON, 1.0)
    return -(safe * np.log(safe)).sum(axis=1) / np.log(N_CLASSES)


def validate_probability_matrix(values: np.ndarray, description: str) -> None:
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[1] != N_CLASSES:
        raise ValueError(f'{description}: probability shape {values.shape} is invalid.')
    if not np.isfinite(values).all() or values.min() < -1e-8 or values.max() > 1 + 1e-8:
        raise ValueError(f'{description}: probability values are invalid.')
    if not np.allclose(values.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError(f'{description}: probability rows do not sum to one.')


def source_key_columns(source_key: str) -> list[str]:
    return ['row_id', 'outer_fold', 'y_silver'] if source_key == 'oof_train' else (['row_id', 'y_silver'] if source_key == 'validation_ensemble' else ['row_id'])


def feature_group(feature_name: str) -> str:
    if feature_name.endswith(('negatif', 'netral', 'positif')):
        return 'class_probability'
    if feature_name.endswith(('confidence', 'entropy')):
        return 'confidence_or_entropy'
    return 'base_model_disagreement'


def verify_base_meta_pair(base: pd.DataFrame, meta: pd.DataFrame, source_key: str, seed: int) -> None:
    key_columns = source_key_columns(source_key)
    expected_n = EXPECTED_ROWS[source_key]
    required_base = set(key_columns + META_FEATURES + ['svm_prediction', 'bert_prediction'])
    required_meta = set(key_columns + META_FEATURES)
    if missing := sorted(required_base.difference(base.columns)):
        raise ValueError(f'Seed {seed}/{source_key}: base columns missing: {missing}')
    if missing := sorted(required_meta.difference(meta.columns)):
        raise ValueError(f'Seed {seed}/{source_key}: meta columns missing: {missing}')
    if len(base) != expected_n or len(meta) != expected_n:
        raise ValueError(f'Seed {seed}/{source_key}: unexpected row count.')
    if base.duplicated(key_columns).any() or meta.duplicated(key_columns).any():
        raise ValueError(f'Seed {seed}/{source_key}: duplicate source keys found.')
    if source_key == 'locked_test_ensemble' and ('y_silver' in base.columns or 'y_silver' in meta.columns):
        raise RuntimeError(f'Seed {seed}: locked-test artifact is not label-blind.')

    base_sorted = base.sort_values(key_columns).reset_index(drop=True)
    meta_sorted = meta.sort_values(key_columns).reset_index(drop=True)
    if not base_sorted[key_columns].equals(meta_sorted[key_columns]):
        raise ValueError(f'Seed {seed}/{source_key}: base and meta keys are not aligned.')
    for feature in META_FEATURES:
        if not np.allclose(base_sorted[feature].to_numpy(float), meta_sorted[feature].to_numpy(float), atol=1e-8, rtol=1e-8):
            raise ValueError(f'Seed {seed}/{source_key}: meta feature {feature} differs from base artifact.')

    svm = base_sorted[SVM_PROB_COLUMNS].to_numpy(float)
    bert = base_sorted[BERT_PROB_COLUMNS].to_numpy(float)
    validate_probability_matrix(svm, f'Seed {seed}/{source_key}/SVM')
    validate_probability_matrix(bert, f'Seed {seed}/{source_key}/IndoBERT')
    if not np.array_equal(base_sorted['svm_prediction'].to_numpy(int), svm.argmax(axis=1)):
        raise ValueError(f'Seed {seed}/{source_key}: SVM hard labels differ from probability argmax.')
    if not np.array_equal(base_sorted['bert_prediction'].to_numpy(int), bert.argmax(axis=1)):
        raise ValueError(f'Seed {seed}/{source_key}: IndoBERT hard labels differ from probability argmax.')
    expected_features = {
        'svm_confidence': svm.max(axis=1),
        'bert_confidence': bert.max(axis=1),
        'svm_entropy': normalized_entropy(svm),
        'bert_entropy': normalized_entropy(bert),
        'pred_disagreement': (svm.argmax(axis=1) != bert.argmax(axis=1)).astype(int),
        'prob_disagreement': 0.5 * np.abs(svm - bert).sum(axis=1),
    }
    for feature, expected in expected_features.items():
        if not np.allclose(base_sorted[feature].to_numpy(float), expected, atol=1e-7, rtol=1e-7):
            raise ValueError(f'Seed {seed}/{source_key}: derived feature {feature} is inconsistent.')


def numeric_summary(values: np.ndarray) -> dict:
    values = np.asarray(values, dtype=float)
    return {
        'n': int(len(values)), 'mean': float(values.mean()), 'std': float(values.std(ddof=1)),
        'p05': float(np.quantile(values, 0.05)), 'p50': float(np.quantile(values, 0.50)),
        'p95': float(np.quantile(values, 0.95)), 'minimum': float(values.min()), 'maximum': float(values.max()),
    }


### Load and validate repeated-seed probability artifacts

Reads the authorized OOF, validation-ensemble, and label-blind locked-test-ensemble base-model outputs. It checks the fixed recipe, five-fold completion, checksums, row counts, and derived meta-features before any distributional comparison.


In [ ]:
protocol_path = R1_06_ROOT / 'protocol_v2.json'
if not protocol_path.is_file():
    raise FileNotFoundError(protocol_path)
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
protocol_hash = protocol.get('protocol_hash')
if protocol.get('planned_seeds') != EXPECTED_SEEDS:
    raise RuntimeError('Base-model planned seeds do not match the fixed post-redesign diagnostic contract.')
if protocol.get('scientific_protocol', {}).get('prediction_generator') != 'five_fold_outer_ensemble_for_validation_and_locked_test':
    raise RuntimeError('Base-model prediction-generator declaration is unexpected.')

SEED_DATA: dict[int, dict[str, pd.DataFrame]] = {}
integrity_rows = []
for seed in EXPECTED_SEEDS:
    seed_dir = R1_06_ROOT / f'seed_{seed}'
    manifest_path = seed_dir / 'run_manifest.json'
    state_path = seed_dir / 'run_state.json'
    schema_path = seed_dir / 'schema.json'
    for path in [manifest_path, state_path, schema_path]:
        if not path.is_file():
            raise FileNotFoundError(path)
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    state = json.loads(state_path.read_text(encoding='utf-8'))
    schema = json.loads(schema_path.read_text(encoding='utf-8'))
    if manifest.get('protocol_hash') != protocol_hash or state.get('protocol_hash') != protocol_hash:
        raise RuntimeError(f'Seed {seed}: protocol hash mismatch.')
    if state.get('status') != 'seed_completed' or manifest.get('fold_count') != 5:
        raise RuntimeError(f'Seed {seed}: Base-model seed is not completed with five folds.')
    if manifest.get('locked_test_metrics_computed') is not False or manifest.get('locked_test_artifact_is_label_blind') is not True:
        raise RuntimeError(f'Seed {seed}: Base-model locked-test guardrail failed.')
    if schema.get('label_order') != LABEL_ORDER or schema.get('meta_feature_names') != META_FEATURES:
        raise RuntimeError(f'Seed {seed}: schema does not match the fixed feature contract.')

    frames = {}
    checksum_ok = True
    for source_key, (base_filename, meta_filename) in SOURCE_FILES.items():
        base_path, meta_path = seed_dir / base_filename, seed_dir / meta_filename
        if not base_path.is_file() or not meta_path.is_file():
            raise FileNotFoundError(f'Seed {seed}: missing {source_key} artifact.')
        for filename, path in [(base_filename, base_path), (meta_filename, meta_path)]:
            expected_hash = manifest.get('output_checksums', {}).get(filename)
            checksum_ok = checksum_ok and expected_hash == sha256_file(path)
        base, meta = pd.read_csv(base_path), pd.read_csv(meta_path)
        verify_base_meta_pair(base, meta, source_key, seed)
        frames[source_key] = base.sort_values(source_key_columns(source_key)).reset_index(drop=True)
    if not checksum_ok:
        raise RuntimeError(f'Seed {seed}: output checksum differs from the base-model manifest.')
    SEED_DATA[seed] = frames
    integrity_rows.append({
        'seed': seed, 'protocol_hash': protocol_hash, 'r1_06_status': state.get('status'),
        'fold_count': manifest.get('fold_count'), 'checksum_validation_passed': checksum_ok,
        'schema_validation_passed': True, 'derived_feature_validation_passed': True,
        'locked_test_label_blind': manifest.get('locked_test_artifact_is_label_blind'),
        'locked_test_metrics_computed_in_r1_06': manifest.get('locked_test_metrics_computed'),
        'oof_train_n': len(frames['oof_train']), 'validation_ensemble_n': len(frames['validation_ensemble']),
        'locked_test_ensemble_n': len(frames['locked_test_ensemble']),
    })

integrity_frame = pd.DataFrame(integrity_rows)
atomic_write_csv(OUTPUT_DIR / 'post_redesign_recipe_integrity_per_seed.csv', integrity_frame)
print(integrity_frame.to_string(index=False))


### Calculate per-seed distribution-shift summaries

Computes probability, confidence, entropy, disagreement, hard-prediction, and distribution-distance summaries across OOF, validation, and locked-test ensemble sources for each seed.


In [ ]:
distribution_rows = []
shift_rows = []
hard_share_rows = []
hard_tvd_rows = []

for seed, frames in SEED_DATA.items():
    for source_key, frame in frames.items():
        for feature in META_FEATURES:
            distribution_rows.append({
                'seed': seed, 'split_source': source_key, 'feature_group': feature_group(feature),
                'feature_name': feature, **numeric_summary(frame[feature].to_numpy(float)),
            })
        for model_key, prob_columns in [('svm', SVM_PROB_COLUMNS), ('indobert', BERT_PROB_COLUMNS)]:
            predictions = frame[prob_columns].to_numpy(float).argmax(axis=1)
            for class_id, class_label in enumerate(LABEL_ORDER):
                hard_share_rows.append({
                    'seed': seed, 'split_source': source_key, 'model_key': model_key,
                    'class_id': class_id, 'class_label': class_label, 'n': len(frame),
                    'predicted_class_n': int((predictions == class_id).sum()),
                    'predicted_class_share': float((predictions == class_id).mean()),
                })

    reference = frames['oof_train']
    for target_key in ['validation_ensemble', 'locked_test_ensemble']:
        target = frames[target_key]
        for feature in META_FEATURES:
            ref_values = reference[feature].to_numpy(float)
            target_values = target[feature].to_numpy(float)
            pooled_sd = np.sqrt((ref_values.var(ddof=1) + target_values.var(ddof=1)) / 2)
            mean_delta = float(target_values.mean() - ref_values.mean())
            shift_rows.append({
                'seed': seed, 'comparison': f'oof_train_to_{target_key}', 'feature_group': feature_group(feature),
                'feature_name': feature, 'reference_n': len(ref_values), 'target_n': len(target_values),
                'reference_mean': float(ref_values.mean()), 'target_mean': float(target_values.mean()),
                'mean_delta_target_minus_oof': mean_delta, 'absolute_mean_delta': abs(mean_delta),
                'pooled_standard_deviation': float(pooled_sd),
                'standardized_mean_difference': float(mean_delta / pooled_sd) if pooled_sd > EPSILON else np.nan,
                'wasserstein_distance': float(wasserstein_distance(ref_values, target_values)),
                'reference_p50': float(np.quantile(ref_values, 0.50)), 'target_p50': float(np.quantile(target_values, 0.50)),
                'reference_p95': float(np.quantile(ref_values, 0.95)), 'target_p95': float(np.quantile(target_values, 0.95)),
            })
        for model_key in ['svm', 'indobert']:
            reference_share = np.array([
                np.mean(reference[SVM_PROB_COLUMNS if model_key == 'svm' else BERT_PROB_COLUMNS].to_numpy(float).argmax(axis=1) == class_id)
                for class_id in range(N_CLASSES)
            ])
            target_share = np.array([
                np.mean(target[SVM_PROB_COLUMNS if model_key == 'svm' else BERT_PROB_COLUMNS].to_numpy(float).argmax(axis=1) == class_id)
                for class_id in range(N_CLASSES)
            ])
            hard_tvd_rows.append({
                'seed': seed, 'comparison': f'oof_train_to_{target_key}', 'model_key': model_key,
                'total_variation_distance_hard_prediction_share': float(0.5 * np.abs(reference_share - target_share).sum()),
                'oof_predicted_negative_share': float(reference_share[0]), 'target_predicted_negative_share': float(target_share[0]),
                'oof_predicted_neutral_share': float(reference_share[1]), 'target_predicted_neutral_share': float(target_share[1]),
                'oof_predicted_positive_share': float(reference_share[2]), 'target_predicted_positive_share': float(target_share[2]),
            })

distribution_summary = pd.DataFrame(distribution_rows).sort_values(['seed', 'split_source', 'feature_name']).reset_index(drop=True)
distribution_shift = pd.DataFrame(shift_rows).sort_values(['comparison', 'feature_name', 'seed']).reset_index(drop=True)
hard_class_shares = pd.DataFrame(hard_share_rows).sort_values(['seed', 'split_source', 'model_key', 'class_id']).reset_index(drop=True)
hard_prediction_tvd = pd.DataFrame(hard_tvd_rows).sort_values(['comparison', 'model_key', 'seed']).reset_index(drop=True)

repeated_shift_summary = (
    distribution_shift.groupby(['comparison', 'feature_group', 'feature_name'], as_index=False)
    .agg(
        seed_n=('seed', 'nunique'),
        wasserstein_mean=('wasserstein_distance', 'mean'), wasserstein_sample_sd=('wasserstein_distance', 'std'),
        absolute_mean_delta_mean=('absolute_mean_delta', 'mean'), absolute_mean_delta_sample_sd=('absolute_mean_delta', 'std'),
        standardized_mean_difference_mean=('standardized_mean_difference', 'mean'),
        standardized_mean_difference_sample_sd=('standardized_mean_difference', 'std'),
    )
    .sort_values(['comparison', 'feature_name']).reset_index(drop=True)
)
hard_tvd_summary = (
    hard_prediction_tvd.groupby(['comparison', 'model_key'], as_index=False)
    .agg(seed_n=('seed', 'nunique'), tvd_mean=('total_variation_distance_hard_prediction_share', 'mean'), tvd_sample_sd=('total_variation_distance_hard_prediction_share', 'std'))
    .sort_values(['comparison', 'model_key']).reset_index(drop=True)
)

for filename, frame in {
    'post_redesign_distribution_summary_per_seed.csv': distribution_summary,
    'post_redesign_oof_ensemble_distribution_shift_per_seed.csv': distribution_shift,
    'post_redesign_oof_ensemble_distribution_shift_repeated_seed_summary.csv': repeated_shift_summary,
    'post_redesign_hard_prediction_class_shares_per_seed.csv': hard_class_shares,
    'post_redesign_hard_prediction_tvd_per_seed.csv': hard_prediction_tvd,
    'post_redesign_hard_prediction_tvd_repeated_seed_summary.csv': hard_tvd_summary,
}.items():
    atomic_write_csv(OUTPUT_DIR / filename, frame)

print('Distribution summaries:', len(distribution_summary))
print('Distribution shifts    :', len(distribution_shift))
print('Hard-label TVD rows    :', len(hard_prediction_tvd))
print(repeated_shift_summary.to_string(index=False))


### Define aggregate diagnostic visualizations

Prepares the plot sources and pooled-value helpers used to visualize probability and uncertainty-feature distributions across the three data sources.


In [ ]:
PLOT_SOURCES = [
    ('oof_train', 'OOF train', '#1f77b4'),
    ('validation_ensemble', 'Validation five-fold ensemble', '#ff7f0e'),
    ('locked_test_ensemble', 'Locked-test five-fold ensemble', '#2ca02c'),
]

def pooled_values(feature: str, source_key: str) -> np.ndarray:
    return np.concatenate([SEED_DATA[seed][source_key][feature].to_numpy(float) for seed in EXPECTED_SEEDS])

fig, axes = plt.subplots(2, 3, figsize=(14.5, 7.2), sharex='col', sharey='row')
for row, (model_name, prefix) in enumerate([('SVM', 'svm'), ('IndoBERT', 'bert')]):
    for column, class_label in enumerate(LABEL_ORDER):
        axis = axes[row, column]
        feature = f'{prefix}_prob_{class_label}'
        for source_key, source_label, color in PLOT_SOURCES:
            axis.hist(pooled_values(feature, source_key), bins=35, range=(0, 1), density=True, histtype='step', linewidth=1.6, color=color, label=source_label)
        axis.set_title(f'{model_name}: P({class_label})')
        axis.set_xlim(0, 1)
        axis.grid(alpha=0.22)
        if row == 1:
            axis.set_xlabel('Predicted probability')
        if column == 0:
            axis.set_ylabel('Density')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False)
fig.suptitle('Post-redesign probability distributions by prediction generator\n(pooled across repeated seeds; descriptive only)', y=1.03)
fig.tight_layout()
probability_figure = FIGURE_DIR / 'post_redesign_probability_distribution_by_split.png'
fig.savefig(probability_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

selected_features = [
    ('svm_confidence', 'SVM confidence'), ('bert_confidence', 'IndoBERT confidence'),
    ('svm_entropy', 'SVM normalized entropy'), ('bert_entropy', 'IndoBERT normalized entropy'),
]
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.6), sharex=True)
for axis, (feature, title) in zip(axes.ravel(), selected_features):
    for source_key, source_label, color in PLOT_SOURCES:
        axis.hist(pooled_values(feature, source_key), bins=35, range=(0, 1), density=True, histtype='step', linewidth=1.6, color=color, label=source_label)
    axis.set_title(title)
    axis.set_xlim(0, 1)
    axis.set_xlabel('Value')
    axis.set_ylabel('Density')
    axis.grid(alpha=0.22)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False)
fig.suptitle('Post-redesign confidence and entropy distributions by prediction generator\n(pooled across repeated seeds; descriptive only)', y=1.03)
fig.tight_layout()
uncertainty_figure = FIGURE_DIR / 'post_redesign_confidence_entropy_distribution_by_split.png'
fig.savefig(uncertainty_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

print('Saved:', probability_figure)
print('Saved:', uncertainty_figure)


### Write diagnostic reports and interpretation guardrails

Writes aggregate diagnostic tables and a manifest that explicitly limits interpretation: the diagnostic documents pipeline transparency and distributional context, not model superiority or a basis for model selection.


In [ ]:
required_outputs = [
    'post_redesign_recipe_integrity_per_seed.csv',
    'post_redesign_distribution_summary_per_seed.csv',
    'post_redesign_oof_ensemble_distribution_shift_per_seed.csv',
    'post_redesign_oof_ensemble_distribution_shift_repeated_seed_summary.csv',
    'post_redesign_hard_prediction_class_shares_per_seed.csv',
    'post_redesign_hard_prediction_tvd_per_seed.csv',
    'post_redesign_hard_prediction_tvd_repeated_seed_summary.csv',
]
if len(integrity_frame) != len(EXPECTED_SEEDS) or not integrity_frame['checksum_validation_passed'].all():
    raise RuntimeError('Post-redesign recipe-integrity checks are incomplete.')
if len(distribution_shift) != len(EXPECTED_SEEDS) * 2 * len(META_FEATURES):
    raise RuntimeError('Unexpected distribution-shift row count.')
if len(hard_prediction_tvd) != len(EXPECTED_SEEDS) * 2 * 2:
    raise RuntimeError('Unexpected hard-prediction TVD row count.')

manifest = {
    'status': 'completed',
    'completed_at_utc': utc_now(),
    'diagnostic_version': 'r1_04b_post_redesign_oof_ensemble_v1',
    'source_artifact': str(R1_06_ROOT.relative_to(PROJECT_ROOT)),
    'protocol_hash': protocol_hash,
    'seeds': EXPECTED_SEEDS,
    'source_semantics': {
        'oof_train': 'Each training row is predicted by its held-out outer-fold model.',
        'validation_ensemble': 'Each row uses the mean probability of the five outer-fold models.',
        'locked_test_ensemble': 'Each row uses the mean probability of the five outer-fold models; labels were not read.',
    },
    'comparison_interpretation': 'Distribution-level comparison only. OOF train and ensemble validation/locked test are different splits and intentionally use different prediction generators, so no per-row probability MAE, correlation, or transition analysis is valid here.',
    'locked_test_guardrails': {
        'label_blind_artifact_verified': True,
        'locked_test_labels_read_by_notebook': False,
        'locked_test_metrics_computed_by_notebook': False,
        'used_for_model_or_seed_selection': False,
    },
    'guardrails': [
        'No model training, refit, calibration fitting, threshold tuning, seed selection, or probability averaging occurs in this diagnostic.',
        'The historical full-refit diagnostic remains preserved and is not overwritten.',
        'This diagnostic measures pipeline transparency after redesign; it is not evidence of model superiority.',
    ],
    'output_checksums': {filename: sha256_file(OUTPUT_DIR / filename) for filename in required_outputs},
    'figure_paths': [str(probability_figure.relative_to(PROJECT_ROOT)), str(uncertainty_figure.relative_to(PROJECT_ROOT))],
}
atomic_write_json(OUTPUT_DIR / 'post_redesign_oof_ensemble_diagnostic_manifest.json', manifest)
print('Post-redesign diagnostic completed. Manifest:', OUTPUT_DIR / 'post_redesign_oof_ensemble_diagnostic_manifest.json')


### Generate post-redesign distribution figures

Creates aggregate probability and confidence or entropy distribution plots for the SVM and IndoBERT five-fold ensembles. These figures are retained because they support the revised supplementary material.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15.2, 8.8), sharex='col', sharey='row')
for row, (model_name, prefix) in enumerate([('SVM', 'svm'), ('IndoBERT', 'bert')]):
    for column, class_label in enumerate(LABEL_ORDER):
        axis = axes[row, column]
        feature = f'{prefix}_prob_{class_label}'
        for source_key, source_label, color in PLOT_SOURCES:
            axis.hist(
                pooled_values(feature, source_key), bins=35, range=(0, 1), density=True,
                histtype='step', linewidth=1.6, color=color, label=source_label,
            )
        axis.set_title(f'{model_name}: P({class_label})', fontsize=12)
        axis.set_xlim(0, 1)
        axis.grid(alpha=0.22)
        if row == 1:
            axis.set_xlabel('Predicted probability')
        if column == 0:
            axis.set_ylabel('Density')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.suptitle('Post-redesign probability distributions by prediction generator', fontsize=16, y=0.975)
fig.text(0.5, 0.935, 'Pooled across repeated seeds; descriptive only', ha='center', va='center', fontsize=11)
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 0.012), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(left=0.065, right=0.99, bottom=0.13, top=0.86, wspace=0.08, hspace=0.33)
probability_figure = FIGURE_DIR / 'post_redesign_probability_distribution_by_split.png'
fig.savefig(probability_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

selected_features = [
    ('svm_confidence', 'SVM confidence'), ('bert_confidence', 'IndoBERT confidence'),
    ('svm_entropy', 'SVM normalized entropy'), ('bert_entropy', 'IndoBERT normalized entropy'),
]
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.6), sharex=True)
for axis, (feature, title) in zip(axes.ravel(), selected_features):
    for source_key, source_label, color in PLOT_SOURCES:
        axis.hist(
            pooled_values(feature, source_key), bins=35, range=(0, 1), density=True,
            histtype='step', linewidth=1.6, color=color, label=source_label,
        )
    axis.set_title(title, fontsize=12)
    axis.set_xlim(0, 1)
    axis.set_xlabel('Value')
    axis.set_ylabel('Density')
    axis.grid(alpha=0.22)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.suptitle('Post-redesign confidence and entropy distributions by prediction generator', fontsize=16, y=0.975)
fig.text(0.5, 0.935, 'Pooled across repeated seeds; descriptive only', ha='center', va='center', fontsize=11)
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 0.012), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(left=0.08, right=0.99, bottom=0.13, top=0.86, wspace=0.16, hspace=0.34)
uncertainty_figure = FIGURE_DIR / 'post_redesign_confidence_entropy_distribution_by_split.png'
fig.savefig(uncertainty_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

print('Regenerated final figure:', probability_figure)
print('Regenerated final figure:', uncertainty_figure)


### Create the English-labelled supplementary figure

Builds the release-ready supplementary diagnostic figure with English display labels and line-style legend handles. The final image is distributed separately through the approved aggregate-output whitelist.


In [ ]:
from matplotlib.lines import Line2D
import matplotlib.image as mpimg

DISPLAY_CLASS_LABELS = {'negatif': 'negative', 'netral': 'neutral', 'positif': 'positive'}
legend_handles = [
    Line2D([0], [0], color=color, linewidth=2.0, label=label)
    for _, label, color in PLOT_SOURCES
]

fig, axes = plt.subplots(2, 3, figsize=(15.2, 8.8), sharex='col', sharey='row')
for row, (model_name, prefix) in enumerate([('SVM', 'svm'), ('IndoBERT', 'bert')]):
    for column, class_label in enumerate(LABEL_ORDER):
        axis = axes[row, column]
        feature = f'{prefix}_prob_{class_label}'
        for source_key, _, color in PLOT_SOURCES:
            axis.hist(pooled_values(feature, source_key), bins=35, range=(0, 1), density=True, histtype='step', linewidth=1.6, color=color)
        axis.set_title(f"{model_name}: P({DISPLAY_CLASS_LABELS[class_label]})", fontsize=12)
        axis.set_xlim(0, 1)
        axis.grid(alpha=0.22)
        if row == 1:
            axis.set_xlabel('Predicted probability')
        if column == 0:
            axis.set_ylabel('Density')
fig.suptitle('Post-redesign probability distributions by prediction generator', fontsize=16, y=0.975)
fig.text(0.5, 0.935, 'Pooled across repeated seeds; descriptive only', ha='center', va='center', fontsize=11)
fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.012), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(left=0.065, right=0.99, bottom=0.13, top=0.86, wspace=0.08, hspace=0.33)
probability_figure = FIGURE_DIR / 'post_redesign_probability_distribution_by_split.png'
fig.savefig(probability_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

selected_features = [
    ('svm_confidence', 'SVM confidence'), ('bert_confidence', 'IndoBERT confidence'),
    ('svm_entropy', 'SVM normalized entropy'), ('bert_entropy', 'IndoBERT normalized entropy'),
]
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.6), sharex=True)
for axis, (feature, title) in zip(axes.ravel(), selected_features):
    for source_key, _, color in PLOT_SOURCES:
        axis.hist(pooled_values(feature, source_key), bins=35, range=(0, 1), density=True, histtype='step', linewidth=1.6, color=color)
    axis.set_title(title, fontsize=12)
    axis.set_xlim(0, 1)
    axis.set_xlabel('Value')
    axis.set_ylabel('Density')
    axis.grid(alpha=0.22)
fig.suptitle('Post-redesign confidence and entropy distributions by prediction generator', fontsize=16, y=0.975)
fig.text(0.5, 0.935, 'Pooled across repeated seeds; descriptive only', ha='center', va='center', fontsize=11)
fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.012), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(left=0.08, right=0.99, bottom=0.13, top=0.86, wspace=0.16, hspace=0.34)
uncertainty_figure = FIGURE_DIR / 'post_redesign_confidence_entropy_distribution_by_split.png'
fig.savefig(uncertainty_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(2, 1, figsize=(15.2, 17.0))
for axis, panel, title, image_path in [
    (axes[0], 'A', 'Probability distributions', probability_figure),
    (axes[1], 'B', 'Confidence and normalized-entropy distributions', uncertainty_figure),
]:
    axis.imshow(mpimg.imread(image_path))
    axis.axis('off')
    axis.text(0.01, 0.985, panel, transform=axis.transAxes, ha='left', va='top', fontsize=15, fontweight='bold', color='black', bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.88, 'pad': 2})
    axis.set_title(title, loc='left', fontsize=13, pad=8)
fig.subplots_adjust(left=0.02, right=0.98, bottom=0.02, top=0.98, hspace=0.10)
supplementary_s3_figure = FIGURE_DIR / 'supplementary_figure_s3_post_redesign_diagnostics.png'
fig.savefig(supplementary_s3_figure, dpi=220, bbox_inches='tight')
plt.close(fig)

manifest_path = OUTPUT_DIR / 'post_redesign_oof_ensemble_diagnostic_manifest.json'
publication_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
publication_manifest['publication_visualization'] = {
    'class_display_labels': DISPLAY_CLASS_LABELS,
    'legend_style': 'Line2D handles represent histogram-outline sources.',
    'supplementary_figure_s3': str(supplementary_s3_figure.relative_to(PROJECT_ROOT)),
    'note': 'Panel A and Panel B are descriptive diagnostics and were not used for model or seed selection.',
}
publication_manifest['figure_paths'] = [
    str(probability_figure.relative_to(PROJECT_ROOT)),
    str(uncertainty_figure.relative_to(PROJECT_ROOT)),
    str(supplementary_s3_figure.relative_to(PROJECT_ROOT)),
]
publication_manifest['figure_checksums'] = {
    probability_figure.name: sha256_file(probability_figure),
    uncertainty_figure.name: sha256_file(uncertainty_figure),
    supplementary_s3_figure.name: sha256_file(supplementary_s3_figure),
}
atomic_write_json(manifest_path, publication_manifest)
print('Publication-ready Supplementary Figure S3:', supplementary_s3_figure)
